# Basic Usage

기본 호출, 시스템 프롬프트, 온도/토큰 제어, 멀티턴 대화, 스트리밍, Thinking 모드.

In [ ]:
from llm_request._internal import handler_client as llm_handler
from llm_request._internal.handler_client import call

# llm_handler.configure(base_url="http://localhost:8100")


### 가장 단순한 호출

In [ ]:
from llm_request._internal.handler_client import call

response = call("대한민국의 수도는?")
print(response.text)


대한민국의 수도는 **서울**입니다.


### 시스템 프롬프트 설정

In [3]:
response = call(
    "자기소개를 해줘",
    system="당신은 키움증권의 ESG 평가 전문 AI입니다. 간결하게 답변하세요.",
)
print(response.text)

반갑습니다. 키움증권의 **ESG 평가 전문 AI**입니다.

기업의 **환경(E)·사회(S)·지배구조(G)** 데이터를 정밀 분석하여 투자 리스크를 진단하고 비재무적 가치를 평가합니다. 객관적인 지표를 바탕으로 신뢰할 수 있는 ESG 인사이트를 제공하겠습니다.


### 온도 및 토큰 제한

In [14]:
# 결정적 응답 (항상 같은 결과)
response = call("1+1은?", temperature=0.1, max_tokens=200)
print("결정적:", response.text)

# 창의적 응답
response = call("1+1은?", temperature=1.5, max_tokens=200)
print("창의적:", response.text)

결정적: 1+1은 **2**입니다.
창의적: 1+1은 **2**입니다.


### 토큰 사용량 확인

In [16]:
response = call("안녕하세요")
print(response.usage)

input_tokens=2 output_tokens=23 total_tokens=209 thinking_tokens=184


---

## 멀티턴 대화 & 스트리밍

### 수동 이력 관리 - `history=`

In [17]:
response = call(
    "2번에 대해 자세히 설명해줘",
    history=[
        {"role": "user", "content": "파이썬의 장점 3가지를 알려줘"},
        {"role": "model", "content": "1. 간결한 문법 2. 풍부한 라이브러리 3. 커뮤니티"},
    ],
)
print(response.text)

파이썬의 두 번째 장점인 **'풍부한 라이브러리(및 생태계)'**에 대해 자세히 설명해 드릴게요.

라이브러리란 특정 기능을 수행하기 위해 미리 작성된 코드의 집합을 말합니다. 파이썬은 "바퀴를 다시 발명할 필요가 없다"는 철학 아래, 거의 모든 분야에서 바로 가져다 쓸 수 있는 도구들이 완벽하게 갖춰져 있습니다.

주요 특징과 예시는 다음과 같습니다.

### 1. 전 세계 개발자가 만든 방대한 저장소 (PyPI)
파이썬 패키지 인덱스(PyPI)에는 수십만 개의 외부 라이브러리가 등록되어 있습니다. 데이터 분석, 웹 개발, 이미지 처리, 업무 자동화 등 본인이 원하는 기능이 있다면 검색을 통해 이미 잘 만들어진 라이브러리를 설치(`pip install`)해서 바로 사용할 수 있습니다.

### 2. 분야별 대표 라이브러리
파이썬이 특정 분야에서 독보적인 위치를 차지하게 된 이유는 강력한 전용 라이브러리들 덕분입니다.

*   **데이터 과학 및 AI:**
    *   **Pandas/NumPy:** 복잡한 수치 계산과 대량의 데이터를 표 형태로 다루는 데 최적화되어 있습니다.
    *   **TensorFlow/PyTorch:** 인공지능(딥러닝) 모델을 만들 때 전 세계에서 가장 많이 사용되는 도구입니다.
    *   **Matplotlib/Seaborn:** 데이터를 그래프나 차트로 시각화하는 데 사용됩니다.
*   **웹 개발:**
    *   **Django/Flask:** 웹사이트의 뒷단(백엔드)을 빠르고 안정적으로 구축할 수 있게 도와줍니다.
*   **업무 자동화 및 스크래핑:**
    *   **Selenium/BeautifulSoup:** 웹사이트의 정보를 자동으로 수집하거나 브라우저를 제어하는 데 필수적입니다.

### 3. "Batteries Included" (배터리 포함) 철학
파이썬은 설치와 동시에 기본적으로 제공되는 **'표준 라이브러리'**가 매우 강력합니다. 외부 라이브러리를 설치하지 않더라도 운영체제 제어, 파일 읽고

### 자동 이력 관리 - `session()`

In [18]:
from llm_request._internal.handler_client import session

sess = session(system="금융 분석 전문가입니다.")

r1 = sess.call("삼성전자 주가 동향을 분석해줘")
print("[턴 1]")
print(r1.text)

r2 = sess.call("원인이 뭘까?")
print("\n[턴 2]")
print(r2.text)

# 이력 확인
print("\n[이력]")
for msg in sess.get_history():
    print(f"  {msg['role']}: {msg['content'][:50]}...")

sess.clear()

[턴 1]
삼성전자(005930)에 대한 최근 주가 동향과 향후 전망을 금융 분석 전문가의 시각에서 정리해 드립니다.

현재 삼성전자의 주가 흐름은 **'AI 반도체 경쟁력 증명'**과 **'거시 경제 변동성'** 사이의 줄타기 국면에 있다고 볼 수 있습니다.

---

### 1. 최근 주가 흐름 및 현황
최근 삼성전자 주가는 8만 원대 중반(8만전자)까지 회복하며 강한 반등세를 보였으나, 최근 미국발 경기 침체 우려와 엔비디아(NVIDIA) 등 글로벌 테크주의 조정 여파로 인해 7만 원대에서 변동성이 커진 상황입니다.

*   **상승 동인:** 2024년 2분기 '어닝 서프라이즈'(영업이익 10.4조 원)를 기록하며 메모리 업황 회복을 입증했습니다.
*   **하락 동인:** 글로벌 경기 둔화 우려, 미국 대선 등 지정학적 리스크, 그리고 HBM(고대역폭 메모리) 공급 시점에 대한 시장의 의구심이 작용하고 있습니다.

### 2. 핵심 분석 포인트 (Key Factors)

#### ① HBM(고대역폭 메모리) 시장 내 입지 (가장 중요)
삼성전자 주가의 향후 향방을 결정지을 가장 큰 변수는 **엔비디아에 대한 HBM3E(5세대) 공급 여부**입니다.
*   현재 SK하이닉스가 독점하고 있는 시장에 삼성전자가 본격적으로 진입(퀄 테스트 통과 및 양산 공급)하게 될 경우, 주가는 강력한 리레이팅(재평가) 구간에 진입할 것입니다.
*   최근 8단 제품에 대한 승인 가능성이 커지면서 하반기 실적 개선 기대감이 높은 상태입니다.

#### ② 파운드리(위탁생산) 및 시스템LSI
*   TSMC와의 격차를 줄이는 것이 숙제입니다. 3나노 이하 선단 공정에서의 수율 안정화와 대형 고객사(구글, 퀄컴 등) 확보 소식이 들려온다면 주가에 강한 탄력을 줄 것입니다.
*   스마트폰(MX) 부문은 갤럭시 Z폴드6/플립6 등 신제품 출시 효과와 AI 폰 시장 선점 효과를 지켜봐야 합니다.

#### ③ 메모리 업황의 질적 개선
*   단순한 수요 증가를 넘어, DDR5, 

### Session 의 function calling round-trip — `send_tool_results()`

function calling 응답을 받으면 `Session.send_tool_results([(id, name, response_dict), ...])` 한 줄로
tool 결과 회신 + 다음 응답 받기.

내부적으로 Session 이 `assistant_message` (모든 part type + thought_signature) 를 history 에 자동 echo
→ 사용자는 mixed paradigm 여부 신경 안 써도 됨.


In [ ]:
from llm_request._internal.handler_client import session

tools = [
    {
        "name": "get_stock_price",
        "description": "주가 조회",
        "parameters": {
            "type": "object",
            "properties": {"ticker": {"type": "string"}},
            "required": ["ticker"],
        },
    }
]

sess = session()

# Turn 1 — 함수 호출 요청 받기
r1 = sess.call("삼성전자(005930) 주가 알려줘", tools=tools)
fc = r1.function_calls[0]
print(f"[Turn 1] 모델: {fc.name}({fc.args}) 호출 요청")

# 에이전트가 함수 실행 (더미)
result = {"price": 87500, "currency": "KRW"}

# Turn 2 — 결과 회신. tools= 도 Turn 1 과 동일하게 전달 (Gemini 가 함수 정의 잃지 않도록).
r2 = sess.send_tool_results([(fc.id, fc.name, result)], tools=tools)
print(f"[Turn 2] {r2.text}")


---
## 3. 스트리밍 - `stream()`

In [19]:
from llm_request._internal.handler_client import stream

for chunk in stream("ESG 평가 기준을 간단히 설명해줘"):
    print(chunk.text, end="", flush=True)

ESG(환경·사회·지배구조) 평가 기준은 기업이 얼마나 지속 가능한 경영을 하는지를 측정하는 지표입니다. 크게 세 가지 영역으로 나뉩니다.

### 1. E (Environmental, 환경)
기업이 환경에 미치는 영향을 평가합니다.
*   **탄소 배출:** 온실가스 배출량 감축 및 탄소 중립 노력
*   **에너지 효율:** 신재생 에너지 사용 및 에너지 절약
*   **자원 관리:** 폐기물 줄이기, 재활용 확대, 수자원 보호
*   **친환경 제품:** 환경에 무해한 제품 개발 및 공급망 관리

### 2. S (Social, 사회)
기업이 직원, 고객, 협력사, 지역사회와 맺는 관계를 평가합니다.
*   **인권 및 노동:** 강제 노동 금지, 아동 노동 금지, 차별 철폐
*   **안전 및 보건:** 직원의 안전한 작업 환경 조성
*   **다양성:** 성별, 인종 등의 다양성을 존중하는 고용 문화
*   **사회 공헌:** 지역사회 기여 및 고객 만족도, 개인정보 보호

### 3. G (Governance, 지배구조)
기업의 의사결정 과정과 투명성을 평가합니다.
*   **이사회 구조:** 이사회의 독립성 및 전문성, 여성 이사 비율
*   **주주 권리:** 주주의 권익 보호 및 배당 정책의 적절성
*   **윤리 경영:** 뇌물 수수 금지, 부패 방지 및 내부 고발 시스템
*   **공시 투명성:** 기업 정보를 정확하고 투명하게 공개하는지 여부

---

**요약하자면:**
*   **E:** 지구를 위해 얼마나 노력하는가?
*   **S:** 사람(직원, 사회)을 얼마나 소중히 여기는가?
*   **G:** 투명하고 깨끗하게 운영되는가?

최근에는 투자자들이 기업의 재무적 성과뿐만 아니라 이러한 **ESG 지표를 보고 기업의 미래 가치와 위험 요소를 판단**하기 때문에 매우 중요해지고 있습니다.

---

## Thinking 모드

복잡한 추론이 필요할 때 Gemini 의 사고 기능을 활성화한다.

### `thinking` 인자

| 레벨 | 사고 깊이 | 토큰 소모 |
| --- | --- | --- |
| `"minimal"` | 최소 | 적음 |
| `"low"` | 낮음 | |
| `"medium"` | 중간 | |
| `"high"` | 깊음 | 많음 |

### ⚠️ 기본값 (`thinking=None` 일 때 모델별 동작)

SDK 가 `thinking` 인자를 명시하지 않으면 (`None`) Gemini 백엔드의 모델별 default 가 적용된다:

| 모델 | 기본 동작 | 비용 영향 |
| --- | --- | --- |
| **Gemini 3 Flash/Pro** | **`HIGH` 자동 활성** | thinking 토큰 대량 사용, latency ↑ |

> **Gemini 3 사용 시 비용 통제** — 단순 분류/추출/요약 작업처럼 깊은 추론이 불필요한 호출에는 명시적으로 `thinking="minimal"` 를 보내야 thinking 토큰 절약 가능. 명시 안 하면 모든 호출이 자동으로 high thinking 으로 동작.

In [9]:
response = call(
    "23 x 47 x 89를 계산하고 소수인지 판별해줘",
    thinking="medium",
)
print(response.text)
print(f"\n[토큰 사용량] {response.usage}")

계산 결과와 소수 판별 여부는 다음과 같습니다.

### 1. 계산 과정
*   **1단계:** $23 \times 47 = 1,081$
*   **2단계:** $1,081 \times 89 = 96,209$

**결과: 96,209**

---

### 2. 소수 판별
**96,209는 소수가 아닙니다.**

**이유:**
소수는 1과 자기 자신만으로 나누어떨어지는 수여야 합니다. 하지만 96,209는 이미 문제에서 제시된 바와 같이 **23, 47, 89를 약수로 가지고 있기 때문**에 소수가 아닌 **합성수**입니다.

[토큰 사용량] input_tokens=21 output_tokens=179 total_tokens=200 thinking_tokens=None


In [14]:
import httpx, json

# Handler에 직접 요청해서 raw JSON 확인
raw = httpx.post(
    "http://localhost:8100/v1/generate",
    json={
        "messages": [{"role": "user", "content": "1+1은?"}],
        "thinking": "high",
    },
).json()

print("=== Raw Handler Response ===")
print(json.dumps(raw, indent=2, ensure_ascii=False))

=== Raw Handler Response ===
{
  "text": "1 + 1은 **2**입니다.",
  "model": "gemini-3.5-flash",
  "usage": {
    "input_tokens": 6,
    "output_tokens": 10,
    "total_tokens": 166,
    "thinking_tokens": 150
  },
  "finish_reason": "STOP",
  "request_id": "b7b56228-beb8-4f28-a5a8-05165e0bb64a"
}
